# Stage 2 - light duplication, label encoding and feature scaling

Turn the split signatures into model-ready arrays. Light duplication is a convergence crutch that only fires on signature-scarce classes; labels are encoded and features scaled per-feature to [0,1], fitting on train only.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Step 1 - reload the split from stage 1

In [ ]:
train, test = {}, {}
for name in DATASETS:
    train[name] = pd.read_csv(config.PROCESSED_DIR / f'{name}_train.csv')
    test[name]  = pd.read_csv(config.PROCESSED_DIR / f'{name}_test.csv')
print('loaded', DATASETS)

## Step 2 - light duplication on the train split only
Note it does nothing on signature-rich ROAD, and pads the scarce CICIoV attack classes to the target.

In [ ]:
from adversec.pipeline import duplicate_train_classes
train_dup = {}
for name in DATASETS:
    cfg = config.load_dataset_config(name)
    train_dup[name] = duplicate_train_classes(train[name], benign_class=cfg['benign_label'])
    fired = 'FIRED' if len(train_dup[name]) != len(train[name]) else 'did not fire'
    print(f'{name:12s} train {len(train[name]):>6,} -> {len(train_dup[name]):>6,}  (duplication {fired})')

## Step 3 - encode labels and scale features (fit on train only)
The scaler is per-feature, so the arbitration ID (wide range) and payload bytes (0-255) are each scaled by their own range.

In [ ]:
from adversec.pipeline import encode_labels, scale_features
import joblib
for name in DATASETS:
    y_tr, y_te, enc = encode_labels(train_dup[name], test[name])
    X_tr, X_te, scaler = scale_features(train_dup[name], test[name], FEATURES)
    print(f'\n=== {name} ===')
    print('  label mapping :', {i: c for i, c in enumerate(enc.classes_)})
    print('  X_train', X_tr.shape, ' X_test', X_te.shape)
    print('  raw ID range  :', int(scaler.data_min_[0]), '->', int(scaler.data_max_[0]))
    np.savez(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz', X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te)
    train_dup[name].to_csv(config.PROCESSED_DIR / f'{name}_train_dup.csv', index=False)
    joblib.dump(scaler, config.PROCESSED_DIR / f'{name}_feature_scaler.joblib')
    joblib.dump(enc, config.PROCESSED_DIR / f'{name}_label_encoder.joblib')
    print('  saved arrays + scaler + encoder')